In [3]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import sys
sys.path.append('/tmp/local_scratch/v_neelesh_bisht/3d-cnn/rsna')

import matplotlib.pyplot as plt
import numpy as np
import torch

from config import ResnetConfig
# from dataset import ImageDataset, MaskDataset
from utils import CommonUtils, Device
from resnet_cams import GradCam, DiffCAM, CounterFactual, TorchCAM
# from models import Resnet3D

ModuleNotFoundError: No module named 'torchcam'

In [ ]:
print(torch.cuda.device_count())
torch.cuda.empty_cache()
config = ResnetConfig()
config.phase = 'test'
config.gpu_id = [2]

# get device for processing
device_obj = Device()
device_obj.set_device(config.gpu_id[0])
device = device_obj.get_device()
print(f"Using device: {device}")

In [ ]:
# get test data and mask data

output_dir = os.path.join(os.getcwd(), "MosMedData")
os.makedirs(output_dir, exist_ok=True)

test_scan_paths = [os.path.join(output_dir, "CT-1", x) for x in sorted(os.listdir(os.path.join(output_dir, "CT-1")))]
mask_scan_paths = [os.path.join(output_dir, "mask", x) for x in sorted(os.listdir(os.path.join(output_dir, "mask")))]
test_scan_paths = test_scan_paths[0:3]
mask_scan_paths = mask_scan_paths[0:3]
x_test = np.array([ImageDataset.process_image(path) for path in test_scan_paths])
y_test = np.array([1 for _ in range(len(test_scan_paths))])

print('Number of samples in test are %d' % (x_test.shape[0]))
## change this index for the sample to test
test_sample_index = 0

test_sample = x_test[test_sample_index] # width, height, depth
test_tensor = torch.tensor(test_sample).permute(2, 1, 0) # depth, height, width

masks = np.array([MaskDataset.process_mask(path) for path in mask_scan_paths])
mask_test_volume = masks[test_sample_index]

In [ ]:
model, parameters = Resnet3D.generate_model(config) 
model.load_state_dict(torch.load(config.model_path, map_location=device))
print(model)

In [ ]:
last_conv_layer_name = 'layer4'
last_conv_layer_name_with_module = 'module.layer4'
sample = test_tensor.unsqueeze(0).unsqueeze(0).to(device, dtype=torch.float32)

target_class_idx = 1  # Target class
ref_class_idx = 0  # Reference class


##The CAMs below are obtained through "torch-cam" 3rd party package.
#FIXME the score cam code dosen't works as of now.
# torch_cam_score_cam_heatmap = TorchCAM.compute_score_cam(sample, model, last_conv_layer_name_with_module, target_class_idx)
# plt.matshow(np.squeeze(torch_cam_score_cam_heatmap[:, :, 3]))
# plt.show()

torch_cam_grad_cam_heatmap = TorchCAM.compute_grad_cam(sample, model, last_conv_layer_name_with_module, target_class_idx)
plt.matshow(np.squeeze(torch_cam_grad_cam_heatmap[:, :, 3]))
plt.show()

torch_cam_grad_campp_heatmap = TorchCAM.compute_grad_campp(sample, model, last_conv_layer_name_with_module, target_class_idx)
plt.matshow(np.squeeze(torch_cam_grad_campp_heatmap[:, :, 3]))
plt.show()


# Generate grad and diff-grad class activation heatmap
grad_cam_heatmap, _,  diff_grad_cam_heatmap= GradCam.compute(sample, model, last_conv_layer_name, target_class_idx, ref_class_idx)
plt.matshow(np.squeeze(grad_cam_heatmap[:, :, 3]))
plt.show()

# Generate diff class activation heatmap
diff_cam_heatmap = DiffCAM.make_diffcam_heatmap(sample, model, target_class_idx, ref_class_idx)
plt.matshow(np.squeeze(diff_cam_heatmap[:, :, 3]))
plt.show()


#Generate counter factual heatmap
counter_factual_heatmap = CounterFactual.make_counter_factual_heatmap(sample, model, target_class_idx, ref_class_idx)
plt.matshow(np.squeeze(counter_factual_heatmap[:, :, 3]))
plt.show()


# Resize heatmap
grad_cam_heatmap = CommonUtils.get_resized_heatmap(grad_cam_heatmap, test_sample.shape)
diff_grad_cam_heatmap = CommonUtils.get_resized_heatmap(diff_grad_cam_heatmap, test_sample.shape)
diff_cam_heatmap = CommonUtils.get_resized_heatmap(diff_cam_heatmap, test_sample.shape)
counter_factual_heatmap = CommonUtils.get_resized_heatmap(counter_factual_heatmap, test_sample.shape)
torch_cam_grad_cam_heatmap = CommonUtils.get_resized_heatmap(torch_cam_grad_cam_heatmap, test_sample.shape)
torch_cam_grad_campp_heatmap = CommonUtils.get_resized_heatmap(torch_cam_grad_campp_heatmap, test_sample.shape)
# torch_cam_score_cam_heatmap = CommonUtils.get_resized_heatmap(torch_cam_score_cam_heatmap, test_sample.shape)

In [ ]:
#TODO move to MaskDataset normalization function
# Assuming mask values are 0 and 1, convert to yellow (1) and black (0)
mask_test = np.where(np.abs(mask_test_volume) < 1e-15, 0, 1)
 # Set non-1 values to NaN for transparency
mask_test = np.where(mask_test == 1, 1, np.nan) 

indices = np.where(mask_test == 1)
indices_list = list(zip(*indices))
arr = [i[2] for i in indices_list]

# Find the range of the slices
min_slice = np.min(np.unique(arr))
max_slice = np.max(np.unique(arr))

# Create a single figure with subplots
fig, ax = plt.subplots(max_slice - min_slice + 1, 8, figsize=(30, 5 * (max_slice - min_slice + 1)))

# Loop through each slice and plot the images in the appropriate subplot
for i in range(min_slice, max_slice + 1):
    slice_idx = i - min_slice

    # Adding titles for the first row
    if slice_idx == 0:
        ax[slice_idx, 0].set_title('Sample')
        ax[slice_idx, 1].set_title('Mask')
        ax[slice_idx, 2].set_title('Grad-CAM')
        ax[slice_idx, 3].set_title('Diff Grad-CAM')
        ax[slice_idx, 4].set_title('Diff-CAM')
        ax[slice_idx, 5].set_title('Counter Factual')
        ax[slice_idx, 6].set_title('Torch-CAM Grad-CAM')
        ax[slice_idx, 7].set_title('Torch-CAM Grad-CAM++')

    ax[slice_idx, 0].imshow(np.squeeze(test_sample[:, :, i]), origin='upper', cmap='bone')
    ax[slice_idx, 1].imshow(np.squeeze(mask_test[:, :, i]), origin='upper', cmap='spring')

    img3 = ax[slice_idx, 2].imshow(np.squeeze(test_sample[:, :, i]), cmap='bone')
    img4 = ax[slice_idx, 2].imshow(np.squeeze(grad_cam_heatmap[:, :, i]), cmap='jet', alpha=0.5, extent=img3.get_extent())
    img5 = ax[slice_idx, 2].imshow(np.squeeze(mask_test[:, :, i]), cmap='spring', alpha=0.7, extent=img3.get_extent())

    img6 = ax[slice_idx, 3].imshow(np.squeeze(test_sample[:, :, i]), cmap='bone')
    img7 = ax[slice_idx, 3].imshow(np.squeeze(diff_grad_cam_heatmap[:, :, i]), cmap='jet', alpha=0.5, extent=img6.get_extent())
    img8 = ax[slice_idx, 3].imshow(np.squeeze(mask_test[:, :, i]), cmap='spring', alpha=0.7, extent=img6.get_extent())

    img9 = ax[slice_idx, 4].imshow(np.squeeze(test_sample[:, :, i]), cmap='bone')
    img10 = ax[slice_idx, 4].imshow(np.squeeze(diff_cam_heatmap[:, :, i]), cmap='jet', alpha=0.5, extent=img9.get_extent())
    img11 = ax[slice_idx, 4].imshow(np.squeeze(mask_test[:, :, i]), cmap='spring', alpha=0.7, extent=img9.get_extent())

    img12 = ax[slice_idx, 5].imshow(np.squeeze(test_sample[:, :, i]), cmap='bone')
    img13 = ax[slice_idx, 5].imshow(np.squeeze(counter_factual_heatmap[:, :, i]), cmap='jet', alpha=0.5, extent=img12.get_extent())
    img14 = ax[slice_idx, 5].imshow(np.squeeze(mask_test[:, :, i]), cmap='spring', alpha=0.7, extent=img12.get_extent())

    img15 = ax[slice_idx, 6].imshow(np.squeeze(test_sample[:, :, i]), cmap='bone')
    img16 = ax[slice_idx, 6].imshow(np.squeeze(torch_cam_grad_cam_heatmap[:, :, i]), cmap='jet', alpha=0.5, extent=img15.get_extent())
    img17 = ax[slice_idx, 6].imshow(np.squeeze(mask_test[:, :, i]), cmap='spring', alpha=0.7, extent=img15.get_extent())

    img18 = ax[slice_idx, 7].imshow(np.squeeze(test_sample[:, :, i]), cmap='bone')
    img19 = ax[slice_idx, 7].imshow(np.squeeze(torch_cam_grad_campp_heatmap[:, :, i]), cmap='jet', alpha=0.5, extent=img18.get_extent())
    img20 = ax[slice_idx, 7].imshow(np.squeeze(mask_test[:, :, i]), cmap='spring', alpha=0.7, extent=img18.get_extent())

# Adjust the layout
plt.tight_layout()
plt.show()

In [ ]:
# Here’s a more detailed color mapping from the jet colormap:

# Blue: Low values (0 to ~0.25)
# Cyan: Mid-low values (~0.25 to ~0.45)
# Green: Mid values (~0.45 to ~0.55) > (~114 to ~140)
# Yellow: High intermediate values (~0.55 to ~0.75)
# Orange: Very high intermediate values (~0.75 to ~0.85)
# Red: Maximum values (~0.85 to 1)

#ASSUMPTIONS of threshold here
HEATMAP_THRESHOLD = 140
MASK_THRESHOLD = 1e-15 # think more on this, or we should train the whole model again from scratch with the same shape of data as that of the shape of mask.

heatmap_obj = {
        "grad_cam_heatmap":grad_cam_heatmap, 
        "diff_grad_cam_heatmap": diff_grad_cam_heatmap,
        "diff_cam_heatmap": diff_cam_heatmap, 
        "counter_factual_heatmap": counter_factual_heatmap,
        "torch_cam_grad_cam_heatmap": torch_cam_grad_cam_heatmap,
        "torch_cam_grad_campp_heatmap": torch_cam_grad_campp_heatmap
    }
gt_mask = np.where(np.abs(mask_test_volume) < MASK_THRESHOLD, 0, 1)  # ground truth mask (3D)


In [ ]:
####################################### Calculate IOU #######################################

def calculate_iou_3d(pred_mask, gt_mask):
    """
    Calculate the Intersection over Union (IoU) of two 3D masks.
    
    Parameters:
    pred_mask: 3D numpy array of predicted mask (binary)
    gt_mask: 3D numpy array of ground truth mask (binary)
    
    Returns:
    float: IoU value
    """
    intersection = np.logical_and(pred_mask, gt_mask)
    union = np.logical_or(pred_mask, gt_mask)
    
    intersection_volume = np.sum(intersection)
    union_volume = np.sum(union)
    
    if union_volume == 0:
        return 0
    
    iou = intersection_volume / union_volume
    
    return iou


for label, heatmap in heatmap_obj.items():
    pred_mask = np.where(heatmap < HEATMAP_THRESHOLD, 0, 1)  # Binary mask from Grad-CAM heatmap
    iou = calculate_iou_3d(pred_mask, gt_mask)
    print(f"{label} IoU:", iou)


In [ ]:
####################################### Calculate Localization Metric, Energy Proportion #######################################

def calculate_energy_proportion(saliency_map_3d, gt_mask):
    """
    Calculate the Localization metric, energy proportion.
    
    Parameters:
    saliency_map_3d: 3D numpy array of the saliency map.
    gt_mask: 3D numpy array of ground truth mask (binary)
    
    Returns:
    float: proportion value
    """
    # Point-wise multiply the saliency map with the mask
    masked_saliency_map = saliency_map_3d * gt_mask

    # Calculate the energy in the bounding box
    energy_in_bbox = np.sum(masked_saliency_map)

    # Calculate the total energy in the saliency map
    total_energy = np.sum(saliency_map_3d)

    # Calculate the proportion of energy in the bounding box
    proportion = energy_in_bbox / total_energy

    return proportion

for label, heatmap in heatmap_obj.items():
    proportion = calculate_energy_proportion(heatmap, gt_mask)
    print(f"{label}, Proportion of energy in bounding box: {proportion:.4f}")

In [ ]:
import nibabel as nib
from scipy.ndimage import zoom
from scipy import ndimage


# # Function to get pixel counts
# def get_pixel_counts(heatmap):
#     unique_values, counts = np.unique(heatmap, return_counts=True)
#     return dict(zip(unique_values, counts))

# # Get pixel counts for Grad-CAM heatmap
# pixel_counts = get_pixel_counts(mask_test_volume)

# # Display the pixel counts
# for value, count in pixel_counts.items():
#     print(f"Value: {value}, Count: {count}")

mask_scan_paths = [os.path.join(output_dir, "mask", x) for x in sorted(os.listdir(os.path.join(output_dir, "mask")))]

for i in range(0, len(mask_scan_paths)-1):
    filepath = mask_scan_paths[i]
    # Read file
    scan = nib.load(filepath)
    # Get raw data
    scan = scan.get_fdata()
    print("before this",np.unique(scan), scan.shape)

# desired_width=128
# desired_height=128
# desired_depth=64

# # Compute zoom factors
# width_factor = desired_width / scan.shape[0]
# height_factor = desired_height / scan.shape[1]
# depth_factor = desired_depth / scan.shape[-1]

# # Rotate volume by 90 degrees
# scan = ndimage.rotate(scan, 90,  axes=(0, 1), reshape=False)

# # Resize the volume using spline interpolated zoom (SIZ)
# scan = zoom(scan, (width_factor, height_factor, depth_factor), order=1)
# print("after this",np.unique(scan))

# print("grad_cam_heatmap",np.unique(grad_cam_heatmap))
# print("pred_mask",np.unique(pred_mask))
# print("mask_test_volume",np.unique(mask_test_volume))
# print("gt_mask",np.unique(gt_mask))

In [ ]:
################################################################################## Quantitaive Metrics ##############################################################################


####################################### IOU #######################################

# mask0
# grad_cam_heatmap IoU: 0.0017339230812455347
# diff_grad_cam_heatmap IoU: 0.03213202223979791
# diff_cam_heatmap IoU: 0.013332849900786457
# counter_factual_heatmap IoU: 0.00024455857177794083
# torch_cam_grad_cam_heatmap IoU: 0.0017339230812455347
# torch_cam_grad_campp_heatmap IoU: 6.251953735542357e-05


# mask1
# grad_cam_heatmap IoU: 0.0033589786474884317
# diff_grad_cam_heatmap IoU: 0.006392894128893492
# diff_cam_heatmap IoU: 0.007398608419528824
# counter_factual_heatmap IoU: 0.00019224607497596923
# torch_cam_grad_cam_heatmap IoU: 0.00335900205582076
# torch_cam_grad_campp_heatmap IoU: 0.0

# mask2
# grad_cam_heatmap IoU: 0.0005450671423616273
# diff_grad_cam_heatmap IoU: 0.0
# diff_cam_heatmap IoU: 0.0025190564988386167
# counter_factual_heatmap IoU: 0.0
# torch_cam_grad_cam_heatmap IoU: 0.0005450671423616273
# torch_cam_grad_campp_heatmap IoU: 0.0



####################################### Localization Evalution #######################################

# mask0
# grad_cam_heatmap, Proportion of energy in bounding box: 0.0110
# diff_grad_cam_heatmap, Proportion of energy in bounding box: 0.0188
# diff_cam_heatmap, Proportion of energy in bounding box: 0.0102
# counter_factual_heatmap, Proportion of energy in bounding box: 0.0119
# torch_cam_grad_cam_heatmap, Proportion of energy in bounding box: 0.0110
# torch_cam_grad_campp_heatmap, Proportion of energy in bounding box: 0.0079

# mask1
# grad_cam_heatmap, Proportion of energy in bounding box: 0.0050
# diff_grad_cam_heatmap, Proportion of energy in bounding box: 0.0068
# diff_cam_heatmap, Proportion of energy in bounding box: 0.0066
# counter_factual_heatmap, Proportion of energy in bounding box: 0.0038
# torch_cam_grad_cam_heatmap, Proportion of energy in bounding box: 0.0050
# torch_cam_grad_campp_heatmap, Proportion of energy in bounding box: 0.0046

# mask2
# grad_cam_heatmap, Proportion of energy in bounding box: 0.0020
# diff_grad_cam_heatmap, Proportion of energy in bounding box: 0.0021
# diff_cam_heatmap, Proportion of energy in bounding box: 0.0018
# counter_factual_heatmap, Proportion of energy in bounding box: 0.0017
# torch_cam_grad_cam_heatmap, Proportion of energy in bounding box: 0.0020
# torch_cam_grad_campp_heatmap, Proportion of energy in bounding box: 0.0010